# **نصب و تنظیمات اولیه**

In [27]:
!pip install -q google-genai pandas openpyxl
!pip -q install pymupdf sentence-transformers faiss-cpu google-genai
import pandas as pd
import numpy as np
import re
from google import genai
from google.genai import types
from google.colab import files, drive
from google import genai
from google.genai import errors, types
import os, glob, time
import faiss
import pymupdf
import sys

In [6]:
uploaded = files.upload()
print(list(uploaded.keys()))

Saving paper.pdf to paper.pdf
['paper.pdf']


In [9]:
KEYS_PATH = " "
LLM_MODEL = "gemini-3.6-flash"
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 6

PDF_PATH = glob.glob("*.pdf")[0]
print("Using PDF:", PDF_PATH)

with open(KEYS_PATH) as f:
    API_KEYS = [line.strip() for line in f if line.strip()]
print(f"{len(API_KEYS)} API keys loaded")

Using PDF: paper.pdf
100 API keys loaded


# **استخراج متن**

In [10]:
def extract_pages(pdf_path):

    doc = pymupdf.open(pdf_path)
    pages = []
    for i, page in enumerate(doc, start=1):
        text = page.get_text().replace("\n", " ")
        text = " ".join(text.split())
        if text:
            pages.append((i, text))
    return pages

def strip_references(pages, marker="REFERENCES"):

    cleaned = []
    for page_no, text in pages:
        pos = text.find(marker)
        if pos != -1:
            before = text[:pos].strip()
            if before:
                cleaned.append((page_no, before))
            break
        cleaned.append((page_no, text))
    return cleaned

pages = strip_references(extract_pages(PDF_PATH))
print(f"{len(pages)} pages kept (last kept page: {pages[-1][0]})")

11 pages kept (last kept page: 11)


## **چانکینگ**

In [11]:
def chunk_pages(pages, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):

    chunks = []
    for page_no, text in pages:
        start = 0
        while start < len(text):
            end = min(start + size, len(text))
            if end < len(text):
                space = text.rfind(" ", start, end)
                if space > start + size // 2:
                    end = space
            chunks.append({"page": page_no, "text": text[start:end].strip()})
            if end >= len(text):
                break
            start = end - overlap
    return chunks

chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks created")

70 chunks created


# **ساخت امبدینگ و ایندکس**

In [13]:
def normalize(x):
    x = np.asarray(x, dtype="float32")
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def embed_documents(texts, batch_size=50):
    vectors = []
    for i in range(0, len(texts), batch_size):
        vectors.extend(llm.embed(texts[i:i + batch_size], "RETRIEVAL_DOCUMENT"))
        time.sleep(1)
        print(f"embedded {min(i + batch_size, len(texts))}/{len(texts)}", end="\r")
    return normalize(vectors)

texts = [c["text"] for c in chunks]
embeddings = embed_documents(texts)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("\nIndex size:", index.ntotal, "| dim:", embeddings.shape[1])

embedded 70/70
Index size: 70 | dim: 768


# **بررسی مدل های امبدینگ**

In [28]:
c = genai.Client(api_key=API_KEYS[0])
for m in c.models.list():
    if "embedContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


# **بازیابی**

In [15]:
def retrieve(question, k=TOP_K):
    q_emb = normalize(llm.embed([question], "RETRIEVAL_QUERY"))
    scores, ids = index.search(q_emb, k)
    return [{**chunks[i], "score": float(s)} for s, i in zip(scores[0], ids[0])]

# **ask پرامپ و تابع**

In [19]:
PROMPT_TEMPLATE = """You are an assistant answering questions about ONE research paper.
Use ONLY the context excerpts below. Each excerpt is tagged with its page number.

Rules:
- Answer directly, without phrases like "Based on the provided context".
- Cite the page numbers you used, e.g. (p. 4).
- Distinguish what THIS paper does from methods or results it merely cites as related work.
- If the context only partially answers the question, give what is supported and state what is missing.
- If the context does not contain the answer at all, reply exactly: "I could not find the answer in the document.Not present in the paper (completely out-of-scope)."

Context:
{context}

Question: {question}

Answer:"""

def ask(question, k=TOP_K, show_sources=True):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[Page {h['page']}] {h['text']}" for h in hits)
    answer = llm.generate(PROMPT_TEMPLATE.format(context=context, question=question))

    print(f"Q: {question}\n")
    print(f"A: {answer}\n")
    if show_sources:
        print("Sources:")
        for h in hits:
            print(f"  - page {h['page']} (score {h['score']:.2f}): {h['text'][:120]}...")
    print("-" * 80)
    return answer

# **تست**

In [20]:
questions = [
    "What is the main goal of this paper?",
    "What is the size of the final dataset, how was it balanced, and what were the sources of the human and machine texts?",
    "Which model did the authors use to generate the machine text, and why did they choose DistilGPT-2? What were the main generation parameters?",
    "How did the Logistic Regression, Random Forest, and XGBoost models perform on BoW and TF-IDF representations, and which model achieved the best result?",
    "What limitations or future work does the paper mention?",
    "When BERT embeddings were used, how did the performance of the traditional models change compared to BoW and TF-IDF?",
    "Who won the FIFA World Cup in 2018?",    # out-of-scope
    "What were the results of the LSTM, CNN, and CNN-LSTM models, and which model performed better in terms of validation accuracy and training speed?",
    "Which model was transferred to the Streamlit web application, and what were the main limitations of the study?",
    "Which classifier performed best, and what accuracy did it achieve?"

    ]

In [21]:
output_file = "rag_results.txt"

class Tee:

    def __init__(self, *files):
        self.files = files

    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()

    def flush(self):
        for f in self.files:
            f.flush()


with open(output_file, "w", encoding="utf-8") as f:
    original_stdout = sys.stdout
    sys.stdout = Tee(sys.stdout, f)

    try:
        for q in questions:
            try:
                ask(q)
            except Exception as e:
                print(f"[error] failed on: {q}\n{e}\n" + "-" * 80)
    finally:
        sys.stdout = original_stdout


Q: What is the main goal of this paper?

A: The main goal of this paper is to distinguish whether a body of text is AI-generated or authored by a human (p. 1). To achieve this, the paper's primary objectives and contributions are:

* Creating and curating a balanced benchmark dataset of 10,000 records containing both human- and AI-generated texts (p. 1, 3, 10).
* Developing, testing, and comparing various vectorization techniques and machine/deep learning models to accurately classify human versus machine-generated text (p. 3, 10).
* Using the SHAP package to make model predictions explainable and validated (p. 1).
* Deploying a TF-IDF Random Forest model to a user-friendly Streamlit web application to allow non-technical users to interact with the detector (p. 1, 3).

Sources:
  - page 3 (score 0.67): re it reaches the maximum audience while social algorithms can be exploited to amplify the message. The platforms where ...
  - page 10 (score 0.66): creation of a novel dataset of 10 00

# **semantic_sim محاسبه**

In [ ]:
CSV_PATH = "/content/drive/MyDrive/RAG.csv"
TXT_PATH = "/content/drive/MyDrive/rag_results.txt"
API_KEYS_PATH = "/content/gemini_api_keys.txt"


with open(API_KEYS_PATH) as f:
    API_KEYS = [line.strip() for line in f if line.strip()]

client = genai.Client(api_key=API_KEYS[0])
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768

def get_embedding(text, task_type="SEMANTIC_SIMILARITY"):
    if pd.isna(text) or text is None or str(text).strip() == "":
        return None
    resp = client.models.embed_content(
        model=EMBED_MODEL,
        contents=[str(text)],
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=EMBED_DIM
        )
    )
    return np.array(resp.embeddings[0].values, dtype="float32")

def cosine_sim(a, b):
    if a is None or b is None:
        return None
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


df = pd.read_csv(CSV_PATH)
print("ستون‌های فعلی CSV:")
print(df.columns.tolist())
print(f"تعداد ردیف‌ها: {len(df)}")


with open(TXT_PATH, "r", encoding="utf-8") as f:
    content = f.read()


pattern = re.compile(
    r"Q:\s*(.*?)\s*\n\s*A:\s*(.*?)\s*(?=Sources:|-{5,}|\Z)",
    re.DOTALL | re.IGNORECASE
)

matches = pattern.findall(content)
print(f"\nتعداد جفت سؤال-جواب پیدا شده در فایل txt: {len(matches)}")


rag_dict = {}
for q, a in matches:
    q_clean = q.strip()
    a_clean = a.strip()
    rag_dict[q_clean] = a_clean

def find_rag_answer(question):

    if question in rag_dict:
        return rag_dict[question]

    for k in rag_dict:
        if question[:50].lower() in k.lower() or k[:50].lower() in question.lower():
            return rag_dict[k]
    return None

df["rag_answer"] = df["question"].apply(find_rag_answer)

print("\nوضعیت پر شدن ستون rag_answer:")
print(df["rag_answer"].notna().value_counts())

print("\nنمونه‌های پر شده:")
for i in range(min(3, len(df))):
    print(f"\n--- سؤال {i+1} ---")
    print("Q:", df.loc[i, "question"][:80])
    ans = df.loc[i, "rag_answer"]
    print("A:", str(ans)[:150] if pd.notna(ans) else "None")


print("\nCalculating semantic similarities with Gemini...\n")

semantic_scores = []
for i, row in df.iterrows():
    try:
        emb_gt  = get_embedding(row["ground_truth"])
        emb_rag = get_embedding(row["rag_answer"])
        sim = cosine_sim(emb_gt, emb_rag)
        semantic_scores.append(round(sim, 4) if sim is not None else None)
        print(f"{i+1:2d}. {str(row['question'])[:55]:<55} → {sim:.4f}" if sim is not None else f"{i+1:2d}. None")
    except Exception as e:
        print(f"{i+1:2d}. ERROR on: {str(row['question'])[:50]} → {e}")
        semantic_scores.append(None)
    time.sleep(0.8)

df["semantic_sim"] = semantic_scores


for col in ["exact_match", "faithfulness", "citation_correct", "notes"]:
    if col not in df.columns:
        df[col] = None


cols = ["question", "ground_truth", "page", "rag_answer",
        "exact_match", "semantic_sim", "faithfulness", "citation_correct", "notes"]
df = df[cols]


print("\n" + "="*90)
print("جدول نهایی (ستون‌های مهم):")
print(df[["question", "semantic_sim"]].to_string())

print("\nآمار semantic_sim:")
print(df["semantic_sim"].describe())

output_csv = "RAG_filled.csv"
output_xlsx = "RAG_filled.xlsx"

df.to_csv(output_csv, index=False, encoding="utf-8-sig")
df.to_excel(output_xlsx, index=False)

print(f"\n✅ فایل‌ها ذخیره شدند:")
print(f"   - {output_csv}")
print(f"   - {output_xlsx}")

files.download(output_csv)
files.download(output_xlsx)